In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, to_date

# S3 Gateway Credentials
S3_ENDPOINT = "http://seaweedfs-s3.seaweedfs.svc.cluster.local:8333"
S3_ACCESS_KEY = "lakefs"
S3_SECRET_KEY = "501QUn9ETMv2tu9V"
BUCKET_NAME = "lake-bronze"

# S3A Paths
BRONZE_PATH = f"s3a://{BUCKET_NAME}/bronze_datalake"
CHECKPOINT_PATH = f"s3a://{BUCKET_NAME}/_checkpoints/structured_streaming_kafka_lp_to_bronze"

# Kafka Configs
KAFKA_BOOTSTRAP_SERVERS = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "kafka-kafka-bootstrap.kafka.svc.cluster.local:9092")
KAFKA_TOPIC = os.getenv("KAFKA_TOPIC", "wearables-lp")

In [2]:
spark = SparkSession.builder \
    .appName("RAW_TO_BRONZE_STREAM") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,"
                                   "org.apache.hadoop:hadoop-aws:3.3.4,"
                                   "io.delta:delta-spark_2.12:3.1.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.S3SingleDriverLogStore") \
    .config("spark.sql.streaming.checkpointFileManagerClass", "org.apache.spark.sql.execution.streaming.FileSystemBasedCheckpointFileManager") \
    .config("spark.jars.ivy", "/tmp/.ivy2") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

Spark Version: 3.5.0


In [3]:
spark.conf.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
spark.conf.set("spark.hadoop.fs.s3a.endpoint", S3_ENDPOINT)
spark.conf.set("spark.hadoop.fs.s3a.access.key", S3_ACCESS_KEY)
spark.conf.set("spark.hadoop.fs.s3a.secret.key", S3_SECRET_KEY)
spark.conf.set("spark.hadoop.fs.s3a.path.style.access", "true")
spark.conf.set("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
spark.conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
spark.conf.set("spark.hadoop.fs.s3a.endpoint.region", "us-east-1")

# Mirroring the configs here guarantees Delta sees them during the checkpoint creation.
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
hadoop_conf.set("fs.s3a.endpoint", S3_ENDPOINT)
hadoop_conf.set("fs.s3a.access.key", S3_ACCESS_KEY)
hadoop_conf.set("fs.s3a.secret.key", S3_SECRET_KEY)
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.endpoint.region", "us-east-1")

In [4]:
raw_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS) \
    .option("subscribe", KAFKA_TOPIC) \
    .option("startingOffsets", "earliest") \
    .load()

bronze_df = raw_stream.select(
    col("key").cast("binary"),
    col("value").cast("string").alias("raw_payload"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp").alias("kafka_timestamp"),
    current_timestamp().alias("ingestion_timestamp"),
    to_date(current_timestamp()).alias("ingestion_date")
)

In [5]:
query = (bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("path", BRONZE_PATH)
    .option("mergeSchema", "true") #.trigger(processingTime='2 seconds')
    .start())

In [ ]:
query.awaitTermination()